<a href="https://colab.research.google.com/github/fmssilva/DL_Proj/blob/main/assignment_1/task1/task1_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pokemon Type Classification | DL Assignment 1

## Task 1 — MLP Baseline

**Goal:** classify 3600 Pokémon images into 9 types using a flat-pixel MLP. This is the *baseline* — intentionally simple, so the gap with CNN (Task 2) is clear and measurable.  

**Metric:** Macro-averaged F1 — used because class imbalance is 2.76× (Water 674 vs Ground 244). Accuracy would be misleading.  


**About the code:**

> All reusable logic lives in `src/`.

> This notebook is the runner + readable story.

> This notebook clones the repo from github, downloads the data from google drive, runs the code cells, during which saves results to the outputs folder and at the end does the automatic download of that folder.

**DO NOT RUN ALL CELLS**

> This notebook requires some 20 minutes to run all training sequences with T4 GPU. So if you don't have those resources, avoid running it all again and **instead just run the needed specific cells you want.**


# Part 0 - Project Load

### Config

In [ ]:
# All run-level constants live HERE.
# Change FAST_RUN to switch between a quick smoke-test and a real Colab run.
# Nothing else needs touching.

# ┌─── FLIP THIS ONE FLAG ─────────────────────────────────────────────────────┐
FAST_RUN = True   # True = smoke-test (tiny data, 2 epochs) | False = real Colab run
# └────────────────────────────────────────────────────────────────────────────┘

# training hyperparams
EPOCHS      = 2     if FAST_RUN else 30
# PATIENCE: val_macro_f1 is noisier than val_loss (small val set → 1 mis-prediction
# swings F1 by ~1.2%). Use PATIENCE=7 to avoid stopping too early on noisy F1 plateaus.
PATIENCE    = 1     if FAST_RUN else 7
LR          = 1e-3
BATCH_SIZE  = 32    if FAST_RUN else 64
IMG_SIZE    = 64    # MLP input: 64x64x3 = 12288 flat features
# NUM_WORKERS: 0 on Windows (spawn multiprocessing duplicates stdout in notebooks),
# 2 on Colab/Linux (fork is safe and speeds up data loading).
import platform as _platform
NUM_WORKERS = 2 if (_platform.system() != "Windows") else 0

# set True on Colab to save outputs zip to Drive after each experiment automatically
SAVE_IN_EACH_RUN = True

# Google Drive folder where outputs zips are stored (Colab only)
DRIVE_OUTPUTS_DIR = "/content/drive/MyDrive/DL_Proj_Outputs"

# smoke-test data size: 9 classes × N = total training images
N_SAMPLES_PER_CLASS = 6   # 54 total — enough to run the full pipeline in seconds

print(f"FAST_RUN    : {FAST_RUN}  (N_SAMPLES_PER_CLASS={N_SAMPLES_PER_CLASS if FAST_RUN else 'all'})")
print(f"EPOCHS      : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"SAVE_IN_EACH_RUN : {SAVE_IN_EACH_RUN}")
print(f"DRIVE_OUTPUTS_DIR: {DRIVE_OUTPUTS_DIR}")


### Project Import and Set Up

In [ ]:
# ── PROJECT IMPORT AND SET UP ─────────────────────────────────────────────────────────

import sys, os, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not os.path.exists("/content/DL_Proj"):
        !git clone https://github.com/fmssilva/DL_Proj.git /content/DL_Proj
    else:
        !git -C /content/DL_Proj pull --ff-only
    os.chdir("/content/DL_Proj/assignment_1")
    %pip install -r requirements.txt -q
else:
    # notebook kernel starts in task1/ — step up to assignment_1/ root
    cwd = Path(os.getcwd())
    if cwd.name.startswith("task"):
        os.chdir(cwd.parent)

ROOT = os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import torch
import pandas as pd
import matplotlib.pyplot as plt

from src.config import (
    SEED, CLASSES, NUM_CLASSES, DATA_DIR, OUT_DIR,
    get_task_out_dir, set_seed,
)

set_seed(SEED)

TASK_OUT_DIR = get_task_out_dir("task1")
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CSV_PATH     = DATA_DIR / "train_labels.csv"
TRAIN_DIR    = DATA_DIR / "Train"
TEST_DIR     = DATA_DIR / "Test"

print("-"*50)
print(f"ROOT        : {ROOT}")
print(f"EPOCHS      : {EPOCHS}  |  PATIENCE : {PATIENCE}  |  LR : {LR}")
print(f"BATCH_SIZE  : {BATCH_SIZE}  |  IMG_SIZE : {IMG_SIZE}x{IMG_SIZE}")
print(f"Device      : {device}  |  PyTorch {torch.__version__}")
print(f"Outputs     : {TASK_OUT_DIR.resolve()}")

### Hot Reload from Git

Hot reaload src/ folder from github without changing this notebook state

So we can do changes locally in src and push to git and don't need to refresh this page and loose the notebook state.

**Attention!!!: we still need to run all the cells that import the new updated files. This code is just to avoid need to do refresh and loosing the notebook state**


In [ ]:
import os
import importlib
import sys
from pathlib import Path

print("Refreshing local changes from GitHub and reloading src modules...")

# Ensure we are in the repository root for git pull
original_cwd = os.getcwd()
repo_path = Path(ROOT).parent # This should be /content/DL_Proj
if repo_path.exists():
    os.chdir(repo_path)
    print(f"Changed directory to: {os.getcwd()}")

    # Perform a git pull
    !git pull --ff-only
    print("Git pull complete.")

    # Navigate back to the original working directory for the notebook
    os.chdir(original_cwd)
    print(f"Changed directory back to: {os.getcwd()}")

    # Dynamically find and reload all modules in src directory
    src_dir = Path(ROOT) / "src"
    if src_dir.exists():
        for py_file in src_dir.rglob("*.py"):
            relative_path = py_file.relative_to(ROOT)
            module_name = str(relative_path).replace(os.sep, '.')[:-3] # Remove .py

            # Only reload if the module has already been imported
            if module_name in sys.modules:
                try:
                    importlib.reload(sys.modules[module_name])
                    print(f"Reloaded: {module_name}")
                except Exception as e:
                    print(f"Error reloading {module_name}: {e}")
            # else: # Uncomment for debugging if you want to see modules not loaded
            #     print(f"Skipping {module_name}, not previously imported.")
    else:
        print(f"Warning: src directory not found at {src_dir}")
else:
    print(f"Error: Repository path not found at {repo_path}")

print("Src code refresh complete. New code from src modules should now be active.")

### Load Data

In [ ]:
# Colab: download zip from Google Drive if not already present.
import zipfile

if Path("data/train_labels.csv").exists():
  print("data/ already present")
else:
    if IN_COLAB:
        %pip install gdown -q
        import gdown
        gdown.download(id="1nVSQZxQubLEPXjSRqGn7rtPzkw-S0zIi", output="data.zip", quiet=False)
        with zipfile.ZipFile("data.zip") as zf:
            zf.extractall("data")
        os.remove("data.zip")
        print("data/ ready")
    else:
        print("ERROR: data/ not found — place it under assignment_1/data/")


df_full = pd.read_csv(CSV_PATH)
print(f"Full dataset : {len(df_full)} rows, {df_full['label'].nunique()} classes")

# FAST_RUN: subsample N per class so every cell finishes fast on CPU
if FAST_RUN:
    df = df_full.groupby("label", group_keys=False).head(N_SAMPLES_PER_CLASS).reset_index(drop=True)
    print(f"FAST_RUN     : subsampled to {len(df)} rows ({N_SAMPLES_PER_CLASS}/class)")
else:
    df = df_full

print(f"\nClass counts used this run:\n{df['label'].value_counts().to_string()}")


---
# Part 1 — Exploratory Data Analysis

EDA always runs on the **full 3600-image training set** (`df_full`), not on the FAST_RUN subset.  
Because we want to understand the real data distribution — using a 54-image subset would distort every plot.  

The train/val split happens later (inside `build_loaders`), strictly after EDA.

**What we check:**
1. Class imbalance — drives loss function choice
2. Visual similarity — predicts which classes will be confused
3. Intra-class variance — predicts per-class difficulty
4. Pixel statistics — confirms whether ImageNet normalisation is a good fit
5. Intensity histogram — channel balance + augmentation motivation
6. PCA / t-SNE — is there any linear separation in flat pixel space?

In [ ]:
# ── EDA Stats — run on the full dataset ───────────────────────────────────────
import src.datasets.eda as eda

print("\n=== Image Size Distribution ===")
size_map = eda.image_size_distribution(TRAIN_DIR)

print("\n=== Data Integrity Check ===")
valid, invalid = eda.check_data_integrity(TRAIN_DIR, df_full)
print(f"Result: {valid} valid, {invalid} invalid")


### Plot 1 — Class Distribution

We might check the imbalance ratio: (max class count ÷ min class count).

A ratio above 2× may bias the model toward majority classes. If so we might use macro_f1 score, class weights in the loss, etc.


In [ ]:
import src.datasets.eda_plots as eda_plots

fig = eda_plots.plot_class_distribution(df_full, out_path=TASK_OUT_DIR / "plots" / "plot_class_distribution.png")
plt.show()
plt.close(fig)


> **Result:**

> Water is the majority class with **674 images (18.7%)**;

> Ground is the minority with **244 images (6.8%)**.

> Imbalance ratio: **2.76×**.  

> This way, if we always predict Water we would get 18.7% accuracy but macro-F1 ≈ 0.021 — this is why we use macro-F1 as the metric, not accuracy.

> For the loss we use inverse-frequency class weighting in CrossEntropyLoss (Ground gets weight ≈1.64×, Rock ≈1.52×, Fighting ≈1.37× vs Water ≈0.59×).  




### Plot 2 — Sample Images per Class

4 random images per class (fixed seed — same grid on every run).

**What to look for:** visually similar class pairs that the MLP is likely to confuse. Note colour and texture similarities across classes (e.g. Bug/Grass both greenish; Fighting/Normal both humanoid). These pairs will show up as off-diagonal errors in the confusion matrix.


In [ ]:
fig = eda_plots.plot_sample_images(TRAIN_DIR, df_full, n_per_class=4, out_path=TASK_OUT_DIR / "plots" / "plot_sample_images.png")
plt.show()
plt.close(fig)


> **Finding — Visual Similarity**

> Three high-risk confusion pairs stand out:  
> 1. **Bug ↔ Grass** — both dominated by green/yellow tones; the MLP flat vector sees near-identical colour histograms.

> 2. **Fighting ↔ Normal** — both humanoid silhouettes. Spatial layout differs (Fighting=muscular pose), but MLP ignores layout — only colour statistics remain, which largely overlap. Confirmed:

> 3. **Ground ↔ Rock** — both have brown/grey palettes with no distinctive hue.

> These are the primary off-diagonal hotspots in the confusion matrix.


### Plot 3 — Average Image per Class

Mean pixel value across all images in each class (after resizing to 64×64).

If we have a crisp average image means the class has consistent appearance (low intra-class variance → easier to classify).

A blurry/washed-out average means high intra-class variance → harder.

In [ ]:

print ("======== Example of Image Avg with a small sub set of images ==========")

small_df = df_full.groupby("label", group_keys=False).head(N_SAMPLES_PER_CLASS).reset_index(drop=True)
fig = eda_plots.plot_average_image_per_class(TRAIN_DIR, small_df, out_path=TASK_OUT_DIR / "plots" / "plot_average_image_per_class.png")
plt.show()
plt.close(fig)


In [ ]:

print ("======== Image Avg of all the data set images ==========")

fig = eda_plots.plot_average_image_per_class(TRAIN_DIR, df_full, out_path=TASK_OUT_DIR / "plots" / "plot_average_image_per_class.png")
plt.show()
plt.close(fig)


> **Results:** as we will see, image sharpness closelly correlate witht the resulting class scores.

> **Sharpest prototypes (low variance → easier):** Fire (warm orange centre, consistent palette → as we will see it indeed gets the best scores: F1=**0.480**, best) and Water (distinctly blue → F1=**0.358**, 2nd best).

> **Blurriest prototypes (high variance → harder):** Normal (washed-out grey — humanoids of all shapes, sizes, and colours → F1=0.199) and Fighting (similar diversity to Normal → F1=0.073, worst class). Ground's average is brownish but indistinct, overlapping Rock → F1=0.088.  



### Plot 4 — Per-Channel Pixel Statistics

Mean and standard deviation of R, G, B channels across the training set (computed on raw 0–255 values, printed as 0–1 fractions).

For transfer-learning models we might want to stick with their normalization values, example from ImageNet, but if we are training our own models and the values differ from ImageNet so we better use our data set values for mean/std.


In [ ]:
fig = eda_plots.plot_pixel_statistics(TRAIN_DIR, df_full, out_path=TASK_OUT_DIR / "plots" / "plot_pixel_statistics.png")
plt.show()
plt.close(fig)


> **Results:**

> This dataset is ~13–15% **brighter** than ImageNet (Pokémon sprites are pastel/bright) and slightly less variable. The difference per channel is < 0.15 — within acceptable range.

> **Conclusion:** we use ImageNet normalisation constants in our transforms. Dataset-specific constants would reduce normalisation error slightly but this way we get ready for better transfer learning models

### Plot 5 — Pixel Intensity Histogram

Histogram of pixel intensity values (0–255) for R, G, B channels sampled from the training set.

We should look for overall brightness range and channel imbalance. A histogram skewed toward high values → bright/pastel dataset. Nearly overlapping R/G/B histograms → low colour diversity. Either pattern suggests augmentation (colour jitter, flip) would help CNNs in Task 2.


In [ ]:
fig = eda_plots.plot_pixel_intensity_histogram(TRAIN_DIR, df_full, n_samples=200, out_path=TASK_OUT_DIR / "plots" / "plot_pixel_intensity_histogram.png")
plt.show()
plt.close(fig)


> **Rsults:**

> All three channels peak in the **180–220 range** — this is a bright/pastel dataset (Pokémon sprites are designed with vivid colours, not photographic statistics).  

> The three histograms largely overlap, indicating moderate colour diversity.

> For MLP (which sees pixels as independent scalars), this overlap means many classes share similar per-pixel intensity ranges — the discriminative signal comes from the combination of R/G/B, not individual channels.  

> For CNN (Task 2): this narrow intensity distribution means colour jitter augmentation would increase training diversity without distorting class-defining hues.


### Plot 6 — PCA → t-SNE Cluster Plot

Flatten each image to 12 288 features → PCA(50 dims) → t-SNE(2D) → scatter coloured by class.

Lets we ehave clear class clusters. If so, MLP has a chance. If everything is mixed together, no amount of FC layers will help, and so we'll need CNNs.

In [ ]:
# ── t-SNE of flat pixel vectors ───────────────────────────────────────────────
# n_per_class=40 → 360 points total — fast enough even on CPU (~10s), meaningful enough to read
# EDA always on full dataset (df_full), never on the FAST_RUN subset
fig = eda_plots.plot_pca_tsne(
    TRAIN_DIR, df_full,
    n_per_class=6 if FAST_RUN else 40,
    out_path=TASK_OUT_DIR / "plots" / "plot_pca_tsne.png",
)
plt.show()
plt.close(fig)


> **Findings:**  

> **No clean clusters** — all 9 classes heavily overlap in t-SNE space. There is no region of 2D pixel-space that belongs exclusively to any class. We won't get good results with MLP. We need spatial structures and CNNs.

### EDA 7 — Dataset Normalisation Constants (from train split only)

Compute the actual pixel mean and std from the **training split rows only**.  
This is the correct way: stats computed on the split you train on, then applied identically to val/test. Computing stats on the full dataset (including val rows) would be a mild form of data leakage.

We also compare against ImageNet constants to confirm how close they are.

**Data Leakage Audit**

| Step | Code location | What data it uses | Safe? |
|---|---|---|---|
| EDA plots (class counts, pixel stats) | Above cells | `df_full` — all labelled images | ✅ EDA is read-only; plots don't affect training |
| Train/val split | `get_train_val_loaders()` → `train_test_split(stratify=label)` | `df` (subsampled or full) | ✅ Split happens **before** any transform or normalisation |
| Normalisation stats | This cell | `df_train_split` rows only | ✅ Computed on train split **after** the split |
| Train loader | `get_train_val_loaders()` | Train split only | ✅ |
| Val loader | `get_train_val_loaders()` | Val split only | ✅ |
| Test loader | `PokemonDataset(TEST_DIR, ...)` | Separate held-out folder | ✅ Never touched until submission |


In [ ]:

# ── Normalisation stats from TRAIN split only ─────────────────────────────────
# We need the split indices first — reuse the same deterministic stratified split
# that get_train_val_loaders uses, so stats are from exactly the right rows.
from sklearn.model_selection import train_test_split as _tts
from src.config import SEED as _SEED, CLASSES as _CLASSES

_label_to_idx  = {c: i for i, c in enumerate(_CLASSES)}
_all_labels    = [_label_to_idx[lbl] for lbl in df["label"]]
_train_idx, _  = _tts(list(range(len(df))), test_size=0.2, random_state=_SEED, stratify=_all_labels)
df_train_split = df.iloc[_train_idx].reset_index(drop=True)

train_mean, train_std = eda_plots.compute_dataset_stats(TRAIN_DIR, df_train_split)

print("Train-split pixel stats (normalised 0-1):")
for i, ch in enumerate(["R", "G", "B"]):
    print(f"  {ch}: mean={train_mean[i]:.4f}, std={train_std[i]:.4f}")
print(f"\nImageNet reference: mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]")
imagenet_mean = [0.485, 0.456, 0.406]
print(f"\nDifference vs ImageNet:")
for i, ch in enumerate(["R", "G", "B"]):
    diff = train_mean[i] - imagenet_mean[i]
    print(f"  {ch}: {diff:+.4f} ({'brighter' if diff > 0 else 'darker'} than ImageNet)")
max_diff = max(abs(train_mean[i] - imagenet_mean[i]) for i in range(3))
print(f"\nConclusion: {'Using ImageNet constants is acceptable (< 0.15 difference).' if max_diff < 0.15 else 'Consider dataset-specific constants.'}")


> **Finding — Normalisation** (cell output above)  
> Train-split mean ≈ **[0.62, 0.58, 0.54]** — approximately **+13–14% brighter** than ImageNet across all channels.  
> Train-split std ≈ **[0.23, 0.22, 0.22]** — similar to ImageNet stds [0.229, 0.224, 0.225]; max difference < 0.01.  
> Since the per-channel mean difference is < 0.15 (acceptable), we use **ImageNet constants** as-is in all transforms — no dataset-specific re-computation needed.


---
# Part 2 — Model Experiments

**Initial Experiment Plan:**


| ID | Name | Model | Criterion (~loss function) | Optimizer | Purpose |
|---|---|---|---|---|---|
| A | vanilla | VanillaMLP (128→64), no BN, no Dropout | CrossEntropy | Adam lr=1e-3 | True bare baseline — no regularisation at all |
| B | mlp_base | MLP (512→256→128), BN, Dropout(0.4) | CE + class weights | Adam lr=1e-3 | First Colab run result (~0.193) |
| C | ls01_drop03 | MLP, Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam lr=1e-3 | Softer targets + lighter dropout |
| D | wd1e4 | MLP, Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | L2 reg on top of C |
| E | sampler | MLP, Dropout(0.3) | CE + label_smoothing=0.1 | Adam, WD=1e-4 | WeightedRandomSampler replaces class weights |
| F | narrow | NarrowMLP (256→128→64→32), Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | Fewer params per layer — less overfit? |
| G | bottleneck | BottleneckMLP (512→1024→256→128), Dropout(0.3) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | Wide middle captures more combinations? |
| H | vanilla_v2 | VanillaMLP_v2 (256→128), no BN, no Dropout | CrossEntropy | Adam lr=1e-3 | A won → more capacity + no reg helps? |
| I | v2_rock_weights | VanillaMLP_v2, no Dropout | CE(Rock×3, Ground×2) | Adam lr=1e-3 | Targeted upweighting of worst 2 classes |
| J | mlp_drop02 | MLP, Dropout(0.2) | CE + weights + label_smoothing=0.1 | Adam, WD=1e-4 | Sweet spot between 0.0 (A) and 0.3 (D)? |
| K | v2_wd1e5 | VanillaMLP_v2, no Dropout | CrossEntropy | Adam, WD=1e-5 | Infinitesimal L2 — tests regularisation boundary |



**rationale:**

> regularisation changes (C–E)

> architecture changes (F–G)

> capacity ablation (H–K)



**what not tried and why**

> More hidden layers / wider layers: we already have overfitting considering the low amount of training data we have. We have train_f1≈0.55 vs val_f1≈0.19, so adding more parameters to do model will make overfitting worse.

> More epochs: we use epochs = 30 and most of models already stop early. And we need too manage ou computing time availability.

> Grid search: a 5×5 grid over {dropout, weight_decay, label_smoothing, LR, architecture} = 25 runs × ~80s ≈ 33 min. We should try a more targeted approach guided by the previous result, more than a blind sweep.


**Preprocessing Pipeline**

> Every image passes through a transform pipeline before reaching the model, implemented in `src/datasets/dataset.py` and selected by `get_train_val_loaders`. The default RGB pipeline is:

```
Raw PNG  →  Resize(64, 64)  →  ToTensor()  →  Normalize(mean, std)
```

| Step | What/Why|
|---|---|
| `Resize(64, 64)` | Bicubic-downsample each image to 64×64 px so we'll have a fixed-size vector to the MLP |
| `ToTensor()` | Converts PIL image uint8 (0–255) to float32 tensors [0, 1] |
| `Normalize(mean, std)` | Per-channel: `pixel = (pixel - mean) / std`, to center around zero and help faster convergence |

> We use ImageNet means `[0.485, 0.456, 0.406]` and stds `[0.229, 0.224, 0.225]`, which are pre-computed over 1.2M images and are a solid approximation for natural images even without fine-tuning, so these values might best fit pre-trained models when we do transfer-learning, and because when we compare this mean/stds from ImageNet to our train-split dataset they are close values, so it is ok for our "learn from scratch models".

**What we don't do**:

> No histogram equalisation in the default pipeline, which would redistribute pixel intensities, and so increase contrast. This would be good to better learn "countours and lines", but would "vanish" edge colors like orange, red, purple, blue, which might be important to distinguish between classes like Fire = orange, Water = blue, Poison = purple.

## Results Manager
> This global dictionary keeps track of the models we train along the notebook, so it is easy to keep track and compare all models we train.

> This global dictionary with models results can be initialized empty, or we can choose to upload some previous output zip folder that we have, so we update this global rsults manager dictionary with all the models we got in previous runs.

In [ ]:
import math
results_tracker: dict = {}


def _print_leaderboard(tracker: dict) -> None:
    """Print all experiments sorted by val_macro_f1 descending."""
    if not tracker:
        return
    rows = sorted(tracker.items(), key=lambda x: x[1]["val_macro_f1"], reverse=True)
    print(f"\n{'Rank':<5} {'Name':<25} {'val_F1':>8} {'val_acc':>8} {'epochs':>7} {'time(s)':>8}")
    print("-" * 63)
    for rank, (name, m) in enumerate(rows, 1):
        print(f"{rank:<5} {name:<25} {m['val_macro_f1']:>8.4f} {m['val_acc']:>8.4f} {m['total_epochs']:>7} {m['train_time_s']:>8.1f}")


def _restore_from_json(results_json_path):
    """Load experiments from a task1_results.json into results_tracker."""
    if results_json_path.exists():
        with open(results_json_path, 'r') as f:
            loaded = json.load(f)
        # support both new key ("experiments") and old key ("all_experiments")
        exps = loaded.get("experiments") or loaded.get("all_experiments") or {}
        results_tracker.update(exps)
        print(f"Restored {len(exps)} experiments from {results_json_path}")
    else:
        print(f"No results JSON found at {results_json_path} — starting fresh.")


# ── On Colab: auto-restore from Drive (no prompt needed) ─────────────────────
from src.evaluation.persistence import restore_from_drive

results_json_path = TASK_OUT_DIR / "results" / "task1_results.json"

restore_from_drive(TASK_OUT_DIR, "task1", ROOT, IN_COLAB, DRIVE_OUTPUTS_DIR)

print("-" * 50)
_restore_from_json(results_json_path)
print("Current experiments in results_tracker:")
_print_leaderboard(results_tracker)


## Setup
These helper cells define the shared infrastructure used by every experiment:
- **`build_loaders(augment, use_sampler, grayscale, equalize)`** — wraps `get_train_val_loaders`, always uses `df` (which can be the subsampled FAST_RUN set)
- **`run_experiment(model, name, criterion, optimizer, scheduler, loaders)`** — full train loop, early stopping on `val_macro_f1`, checkpoint save, evaluate best, store in `results_tracker` and immediately upsert to `task1_results.json`

Each experiment cell is then just 4–5 lines: instantiate model, criterion, optimizer, scheduler → call `run_experiment`.


In [ ]:

# ── Shared imports for all experiment cells ───────────────────────────────────
import torch.nn as nn
from torch.utils.data import DataLoader
from src.datasets.dataset import (
    PokemonDataset, compute_class_weights,
    get_base_transforms, get_gray_transforms, get_train_val_loaders,
)
from src.models.mlp import MLP, VanillaMLP, VanillaMLP_v2, NarrowMLP, BottleneckMLP, WiderMLP, DeepMLP
from src.training.train import train_one_epoch, evaluate
from src.training.early_stopping import EarlyStopping
from src.evaluation.metrics import classification_report_str
from src.evaluation.plots import plot_history, plot_confusion_matrix
from src.evaluation.persistence import save_experiment_result, save_all_results, save_to_drive
from src.evaluation.submission import generate_submission_from_preds, validate_submission

# build class weights from the run's df (correct proportions even for FAST_RUN)
label_to_idx     = {cls: i for i, cls in enumerate(CLASSES)}
_all_train_labels = [label_to_idx[lbl] for lbl in df["label"]]
class_weights    = compute_class_weights(_all_train_labels).to(device)

# test loader is always the full test set — not touched until submission
test_ds     = PokemonDataset(TEST_DIR, get_base_transforms(IMG_SIZE), csv_path=None)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# single path for incremental per-experiment saves and final save_all_results
RESULTS_PATH = TASK_OUT_DIR / "results" / "task1_results.json"
SUB_PATH     = TASK_OUT_DIR / "results" / "submission_task1.csv"

print(f"Test images  : {len(test_ds)}")
print(f"Class weights: { {cls: round(class_weights[i].item(), 3) for i, cls in enumerate(CLASSES)} }")


# ── Helper: build_loaders ─────────────────────────────────────────────────────
def build_loaders(augment: bool = False, use_sampler: bool = False,
                  grayscale: bool = False, equalize: bool = False):
    """Return (train_loader, val_loader) for the current df and hyperparams.
    grayscale=True → (1,H,W) tensors, input_dim=4096. equalize=True → histogram stretch before gray."""
    return get_train_val_loaders(
        CSV_PATH, TRAIN_DIR, IMG_SIZE, BATCH_SIZE,
        augment=augment, use_sampler=use_sampler,
        num_workers=NUM_WORKERS, df_override=df,
        grayscale=grayscale, equalize=equalize,
    )


# pre-build the two standard loader pairs used by all Part 2 experiments
loaders_base    = build_loaders(augment=False, use_sampler=False)
loaders_sampler = build_loaders(augment=False, use_sampler=True)


# ── Helper: _make_test_loader ─────────────────────────────────────────────────
def _make_test_loader(stem: str) -> DataLoader:
    """Return a test DataLoader with the correct transform for the given experiment stem.
    Gray experiments need grayscale transforms; all others use the standard RGB transform."""
    if stem.endswith("_gray_eq"):
        t = get_gray_transforms(IMG_SIZE, equalize=True)
    elif stem.endswith("_gray"):
        t = get_gray_transforms(IMG_SIZE, equalize=False)
    else:
        return test_loader   # reuse the global RGB test loader
    ds = PokemonDataset(TEST_DIR, t, csv_path=None)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


# ── Helper: _is_new_overall_best ─────────────────────────────────────────────
def _is_new_overall_best(name: str, f1: float) -> bool:
    """True if this experiment (solo or ensemble) beats every other entry in results_tracker.
    Compares against all existing entries so the CSV is only updated when we truly have a new top."""
    other_f1s = [v["val_macro_f1"] for k, v in results_tracker.items() if k != name]
    return f1 > max(other_f1s, default=-1)


# ── Helper: run_experiment ────────────────────────────────────────────────────
def run_experiment(model, name: str, criterion, optimizer, scheduler, loaders) -> tuple:
    """
    Train model up to EPOCHS with early stopping on val macro-F1.
    Saves best checkpoint to task1/outputs/checkpoints/<name>.pth.
    Upserts this experiment into task1_results.json after each run.
    If this experiment is the new overall best (beats solo models AND ensembles),
    auto-regenerates submission_task1.csv.
    Returns (model_with_best_weights, history_dict).
    """
    train_loader, val_loader = loaders
    ckpt_path = str(TASK_OUT_DIR / "checkpoints" / f"{name}.pth")
    stopper   = EarlyStopping(patience=PATIENCE, checkpoint_path=ckpt_path)
    history   = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": []}
    t0        = time.time()

    print(f"\n{'='*60}")
    print(f"  Experiment: {name}")
    print(f"  Model params: {sum(p.numel() for p in model.parameters()):,}")
    print(f"{'='*60}")

    for epoch in range(1, EPOCHS + 1):
        ep_t0         = time.time()
        train_loss    = train_one_epoch(model, train_loader, criterion, optimizer, device)
        train_metrics = evaluate(model, train_loader, criterion, device)
        val_metrics   = evaluate(model, val_loader,   criterion, device)
        elapsed       = time.time() - ep_t0
        scheduler.step()

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_metrics["loss"])
        history["train_f1"].append(train_metrics["macro_f1"])
        history["val_f1"].append(val_metrics["macro_f1"])

        print(
            f"  Epoch {epoch:02d}/{EPOCHS} | "
            f"train_loss={train_loss:.4f}  train_f1={train_metrics['macro_f1']:.4f} | "
            f"val_loss={val_metrics['loss']:.4f}  val_f1={val_metrics['macro_f1']:.4f} | "
            f"{elapsed:.1f}s"
        )

        # negate val_f1 because EarlyStopping minimises — this saves the peak-F1 checkpoint
        stopper(-val_metrics["macro_f1"], model)
        if stopper.stop:
            print(f"  Early stopping at epoch {epoch} (patience={PATIENCE}, metric: val_macro_f1).")
            break

    total_time = time.time() - t0

    # reload best checkpoint and run a final clean evaluation
    model.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=True))
    best = evaluate(model, val_loader, criterion, device)

    entry = {
        "val_macro_f1": round(best["macro_f1"], 4),
        "val_acc":      round(best["acc"], 4),
        "val_loss":     round(best["loss"], 4),
        "total_epochs": len(history["train_loss"]),
        "train_time_s": round(total_time, 1),
        "history":      history,
    }
    results_tracker[name] = entry

    print(f"\n  Best checkpoint: val_loss={best['loss']:.4f}  val_macro_f1={best['macro_f1']:.4f}")
    print(f"  Total time: {total_time:.1f}s ({total_time/len(history['train_loss']):.1f}s/epoch)")
    _print_leaderboard(results_tracker)

    # save this experiment's result to disk immediately — no data lost if Colab disconnects
    save_experiment_result(name, entry, RESULTS_PATH)

    # auto-update submission if this solo model beats everyone (solo + ensemble)
    if _is_new_overall_best(name, entry["val_macro_f1"]):
        print(f"\n  New overall best — regenerating submission CSV...")
        _tl = _make_test_loader(name)
        test_preds = []
        model.eval()
        with torch.no_grad():
            for imgs, _ in _tl:
                imgs = imgs.to(device)
                test_preds.extend(model(imgs).argmax(dim=1).cpu().tolist())
        generate_submission_from_preds(_tl, test_preds, CLASSES, SUB_PATH)
        validate_submission(SUB_PATH)
        print(f"  Submission updated -> {SUB_PATH}")

    if SAVE_IN_EACH_RUN:
        save_to_drive(TASK_OUT_DIR, "task1", ROOT, IN_COLAB, DRIVE_OUTPUTS_DIR)

    return model, history


## Experiments Registry


In [ ]:
# Registry: one entry per experiment stem.
# "model"      — zero-arg factory for the RGB version (in_channels=3).
# "model_gray" — zero-arg factory for the grayscale version (in_channels=1).
# "criterion"  — zero-arg factory; must match what was used during training.
#
# Grayscale (_gray, _gray_eq) and augmented (_augmented) variants are registered
# dynamically in Part 3 cells — their stems depend on best_name at runtime.
model_registry = {
    # ── RGB experiments ────────────────────────────────────────────────────────
    "A_vanilla":         {"model":      lambda: VanillaMLP(img_size=IMG_SIZE),
                          "model_gray": lambda: VanillaMLP(img_size=IMG_SIZE, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss()},
    "B_mlp_base":        {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.4),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.4, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights)},
    "C_ls01_drop03":     {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "D_wd1e4":           {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "E_sampler":         {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(label_smoothing=0.1)},
    "F_narrow":          {"model":      lambda: NarrowMLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: NarrowMLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "G_bottleneck":      {"model":      lambda: BottleneckMLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: BottleneckMLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "H_vanilla_v2":      {"model":      lambda: VanillaMLP_v2(img_size=IMG_SIZE),
                          "model_gray": lambda: VanillaMLP_v2(img_size=IMG_SIZE, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss()},
    "I_v2_rock_weights": {"model":      lambda: VanillaMLP_v2(img_size=IMG_SIZE),
                          "model_gray": lambda: VanillaMLP_v2(img_size=IMG_SIZE, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(
                              weight=torch.tensor([1.,1.,1.,2.,1.,1.,3.,1.,1.], device=device))},
    "J_mlp_drop02":      {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.2),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.2, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "K_v2_wd1e5":        {"model":      lambda: VanillaMLP_v2(img_size=IMG_SIZE),
                          "model_gray": lambda: VanillaMLP_v2(img_size=IMG_SIZE, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss()},
    # ── Extension experiments L-N (uncommented after first Colab run) ──────────
    "L_wider_ls":        {"model":      lambda: WiderMLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: WiderMLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "M_c_sampler":       {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(label_smoothing=0.1)},
    "N_cosine_lr":       {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    # ── Extension experiments O-S (5 new runs around C and M) ─────────────────
    "O_c_sampler_cw":    {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "P_drop015_ls":      {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.15),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.15, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},
    "Q_wider_sampler":   {"model":      lambda: WiderMLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: WiderMLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(label_smoothing=0.1)},
    "R_ls015_drop03":    {"model":      lambda: MLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: MLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)},
    "S_deep_ls":         {"model":      lambda: DeepMLP(img_size=IMG_SIZE, dropout=0.3),
                          "model_gray": lambda: DeepMLP(img_size=IMG_SIZE, dropout=0.3, in_channels=1),
                          "criterion":  lambda: nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)},

    # ── Grayscale and augmented variants — registered dynamically in Part 3 ───
    # "{best_name}_gray"       added after Part 3.1 Gray experiment
    # "{best_name}_gray_eq"    added after Part 3.1 Gray+Eq experiment
    # "{best_name}_augmented"  added after Part 3.2 augmentation experiment
}


### A: Vanilla Baseline
###### ──────────────────────────────────────────

 > Tiny 2-layer MLP, no BN, no Dropout, no class weights — a true baseline.

 > Architecture: Flatten -> FC(12288->128) -> ReLU -> FC(128->64) -> ReLU -> FC(64->9)

 > Uses VanillaMLP from src.models.mlp (imported in shared cell above).

 > Purpose: have a basic model to confirm everything works and from which we can start improving

In [ ]:

# This instantiates our model and moves it to device (gpu/cpu)
model_A     = VanillaMLP(img_size=IMG_SIZE).to(device)

# This defines the loss functin
criterion_A = nn.CrossEntropyLoss()   # no class weights — true bare baseline

# This defines the optimizer for gradient descent
optimizer_A = torch.optim.Adam(model_A.parameters(), lr=LR)

# This defines a learning rate scheduler to dynamically adjust the LR during training
scheduler_A = torch.optim.lr_scheduler.StepLR(optimizer_A, step_size=5, gamma=0.5)

model_A, history_A = run_experiment(model_A, "A_vanilla", criterion_A, optimizer_A, scheduler_A, loaders_base)


### B: MLP Baseline
###### ──────────────────────────────────────────

 > 3 FC Layers: (FC → BN → ReLU → Dropout(0.4) → weighted CrossEntropy).

 > Flatten 64×64×3 → 12 288

 > 3 FC to add depth/capacity, but 3 is enough;

 > Width 512 → 256 → 128 to funel and force compression;

 > Batch Norm after every FC to stabilise gradients with large flat input

 > Dropout (tested 0.0–0.4) for regularisation for large param count on ~2880 train images

 > Output - no softmax: `CrossEntropyLoss` applies log-softmax internally

> Loss - CrossEntropyLoss(weight=class_weights) to correct the imbalance

> Optimiser (Adam lr=1e-3, adaptive LR) - robust default

> Scheduler (StepLR(step_size=5, γ=0.5)) - halves LR every 5 epochs

> Early stopping (patience=7) - a bit bigger patience to prevent premature stopping because we are using val_macro_f1, which is noisier than val_loss (80 samples/class → 1 error = ±1.2% swing)

In [ ]:
model_B     = MLP(img_size=IMG_SIZE, dropout=0.4).to(device)
criterion_B = nn.CrossEntropyLoss(weight=class_weights)
optimizer_B = torch.optim.Adam(model_B.parameters(), lr=LR)
scheduler_B = torch.optim.lr_scheduler.StepLR(optimizer_B, step_size=5, gamma=0.5)

model_B, history_B = run_experiment(model_B, "B_mlp_base", criterion_B, optimizer_B, scheduler_B, loaders_base)


### C: Dropout(0.3) + label_smoothing=0.1
###### ──────────────────────────────────────────

> slightly less regularisation (dropout 0.4→0.3).

> softer targets (label_smoothing=0.1) to distribute  0.1 of the probability mass to wrong classes, to prevent the model from trying to be overconfident — known to help imbalanced multi-class problems. So instead of trying to say that a pokemon is water type with 100% confidence: `[1, 0, 0, 0, 0, 0, 0, 0, 0]` it becomes `[0.9, 0.0125, 0.0125, 0.0125, 0.0125, 0.0125, 0.0125, 0.0125, 0.0125]`, and so it improves generalization.



In [ ]:
model_C     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_C = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_C = torch.optim.Adam(model_C.parameters(), lr=LR)
scheduler_C = torch.optim.lr_scheduler.StepLR(optimizer_C, step_size=5, gamma=0.5)

model_C, history_C = run_experiment(model_C, "C_ls01_drop03", criterion_C, optimizer_C, scheduler_C, loaders_base)


### D: + weight_decay=1e-4
###### ──────────────────────────────────────────

> Adds L2 regularisation to Experiment C. L2 penalises large weights, which should help reduce the train/val gap further.


In [ ]:
model_D     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_D = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_D = torch.optim.Adam(model_D.parameters(), lr=LR, weight_decay=1e-4)
scheduler_D = torch.optim.lr_scheduler.StepLR(optimizer_D, step_size=5, gamma=0.5)

model_D, history_D = run_experiment(model_D, "D_wd1e4", criterion_D, optimizer_D, scheduler_D, loaders_base)


### E: WeightedRandomSampler
###### ──────────────────────────────────────────

> We use the `build_loaders(..., use_sampler=True)` so the loader will oversample minority classes. It replaces the class weights in the loss function - we don't use both to not overcorrect the class imbalance.

> we keep label_smoothing from Experiment C, which was independently helpful.

In [ ]:
model_E     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_E = nn.CrossEntropyLoss(label_smoothing=0.1)  # no class weights — sampler handles it
optimizer_E = torch.optim.Adam(model_E.parameters(), lr=LR, weight_decay=1e-4)
scheduler_E = torch.optim.lr_scheduler.StepLR(optimizer_E, step_size=5, gamma=0.5)

model_E, history_E = run_experiment(model_E, "E_sampler", criterion_E, optimizer_E, scheduler_E, loaders_sampler)


### F: Narrow + Deep MLP
###### ──────────────────────────────────────────

> More deep arhcitecture with 4 hidden layers and max width 256:

> `input -> 256 -> 128 -> 64 -> 32 -> 9`

> We're building on the hypothesis that fewer params per layer → less overfit on 2880 train images.

> We keep the best regularisation combo from D: dropout=0.3 + label_smoothing + weight_decay.



In [ ]:
model_F     = NarrowMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_F = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_F = torch.optim.Adam(model_F.parameters(), lr=LR, weight_decay=1e-4)
scheduler_F = torch.optim.lr_scheduler.StepLR(optimizer_F, step_size=5, gamma=0.5)

model_F, history_F = run_experiment(model_F, "F_narrow", criterion_F, optimizer_F, scheduler_F, loaders_base)


### G: Bottleneck MLP
###### ──────────────────────────────────────────

> Architecture: compress and expand again:

> `input (12.288) -> 512 -> 1024 -> 256 -> 128 -> 9`

> Hypothesis: the wide middle (1024 units) captures cross-pixel combinations before compressing down — the bottleneck pattern.

> Same regularisation as D/F — isolating the architecture change.


In [ ]:

model_G     = BottleneckMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_G = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_G = torch.optim.Adam(model_G.parameters(), lr=LR, weight_decay=1e-4)
scheduler_G = torch.optim.lr_scheduler.StepLR(optimizer_G, step_size=5, gamma=0.5)

model_G, history_G = run_experiment(model_G, "G_bottleneck", criterion_G, optimizer_G, scheduler_G, loaders_base)


### H: VanillaMLP_v2 (wider, no regularisation)
###### ──────────────────────────────────────────

> Architecture:

> `12288 -> 256 -> 128 -> 9`

> Hypothesis: VanillaMLP (A) won on the first runs over all regularised models, suggesting the problem is NOT overfitting from width but from regularisation suppressing signal.

> VanillaMLP_v2 doubles the first layer (128->256) to test if more capacity without regularisation continues to help. So `A's ~1.58M params` to this one `~3.18M params`.


In [ ]:
model_H     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
criterion_H = nn.CrossEntropyLoss()   # no class weights — keep same bare baseline setup as A
optimizer_H = torch.optim.Adam(model_H.parameters(), lr=LR)
scheduler_H = torch.optim.lr_scheduler.StepLR(optimizer_H, step_size=5, gamma=0.5)

model_H, history_H = run_experiment(model_H, "H_vanilla_v2", criterion_H, optimizer_H, scheduler_H, loaders_base)


### I: VanillaMLP_v2 + targeted class weights (Rock + Ground)
###### ──────────────────────────────────────────

> Observation: `Rock (F1=0.00)` and `Ground (F1=0.05)` are the worst classes.

> Lets use a custom weight tensor:

> `[Rock weight=3.0, Ground weight=2.0, others=1.0]`

> to see if we add more weight to these 2 hard classes may help, while keeping the zero-regularisation approach that won in A and H.



In [ ]:

weights_I = torch.ones(NUM_CLASSES, device=device)
weights_I[CLASSES.index("Rock")]   = 3.0
weights_I[CLASSES.index("Ground")] = 2.0
print(f"Exp I weights: { {cls: round(weights_I[i].item(), 1) for i, cls in enumerate(CLASSES)} }")

model_I     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
criterion_I = nn.CrossEntropyLoss(weight=weights_I)
optimizer_I = torch.optim.Adam(model_I.parameters(), lr=LR)
scheduler_I = torch.optim.lr_scheduler.StepLR(optimizer_I, step_size=5, gamma=0.5)

model_I, history_I = run_experiment(model_I, "I_v2_rock_weights", criterion_I, optimizer_I, scheduler_I, loaders_base)


### J: MLP with lighter dropout (0.2)
###### ──────────────────────────────────────────

> Previous MLP experiments used dropout=0.4 (B) or 0.3 (C/D/E/F/G).

> VanillaMLP in previous runs won with 0.0 dropout. So maybe the optimal dropout is somewhere between 0.0 and 0.3. We try 0.2 — still has regularisation but much lighter.

> Include best reg combo from D: label_smoothing + weight_decay + class weights.

In [ ]:
model_J     = MLP(img_size=IMG_SIZE, dropout=0.2).to(device)
criterion_J = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_J = torch.optim.Adam(model_J.parameters(), lr=LR, weight_decay=1e-4)
scheduler_J = torch.optim.lr_scheduler.StepLR(optimizer_J, step_size=5, gamma=0.5)

model_J, history_J = run_experiment(model_J, "J_mlp_drop02", criterion_J, optimizer_J, scheduler_J, loaders_base)


### K: VanillaMLP_v2 + tiny L2 (weight_decay=1e-5)
###### ──────────────────────────────────────────

> VanillaMLP (A) and VanillaMLP_v2 (H) have zero regularisation and won in previous runs.

> Lets add some small small L2 penalty (1e-5) (essentially zero) to see if it improves marginal stability on larger weights, without harming.

In [ ]:

model_K     = VanillaMLP_v2(img_size=IMG_SIZE).to(device)
criterion_K = nn.CrossEntropyLoss()   # keep bare loss — only add weight_decay
optimizer_K = torch.optim.Adam(model_K.parameters(), lr=LR, weight_decay=1e-5)
scheduler_K = torch.optim.lr_scheduler.StepLR(optimizer_K, step_size=5, gamma=0.5)

model_K, history_K = run_experiment(model_K, "K_v2_wd1e5", criterion_K, optimizer_K, scheduler_K, loaders_base)


### L: WiderMLP (1024 first layer) + winning C recipe
###### ──────────────────────────────────────────

> C_ls01_drop03 is winning with:

> `(512-wide first layer, LS=0.1 + class_weights + Dropout=0.3)`

> So lets explore variations of it.  Lets double first layer to 1024 neurons gives more capacity to learn colour.

In [ ]:
model_L     = WiderMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_L = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_L = torch.optim.Adam(model_L.parameters(), lr=LR)
scheduler_L = torch.optim.lr_scheduler.StepLR(optimizer_L, step_size=5, gamma=0.5)

print(f"WiderMLP params: {sum(p.numel() for p in model_L.parameters()):,}")
model_L, history_L = run_experiment(
    model_L, "L_wider_ls", criterion_L, optimizer_L, scheduler_L, loaders_base
)

### M: C architecture + WeightedRandomSampler (no loss weights)
###### ──────────────────────────────────────────

> C uses: LS=0.1 + class_weights in loss.

> E uses: LS=0.1 + WeightedRandomSampler (no class weights in loss).

> Now lets test: LS=0.1 + sampler only (loss is unweighted CrossEntropy + LS).

> Sampler ensures each batch is balanced at the data level and so we don't do class weights for the loss function to avoid over-correct and double-penalising minority classes and majority-class accuracy.

In [ ]:
model_M     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_M = nn.CrossEntropyLoss(label_smoothing=0.1)   # sampler handles imbalance
optimizer_M = torch.optim.Adam(model_M.parameters(), lr=LR)
scheduler_M = torch.optim.lr_scheduler.StepLR(optimizer_M, step_size=5, gamma=0.5)

model_M, history_M = run_experiment(model_M, "M_c_sampler", criterion_M, optimizer_M, scheduler_M, loaders_sampler)


### N: C architecture + CosineAnnealingLR
###### ──────────────────────────────────────────

> StepLR(step=5, gamma=0.5) drops LR abruptly at epochs 5, 10, 15, 20. This causes sudden changes in the loss landscape → val_f1 oscillations → early stopping may terminate before the model has fully settled.

> CosineAnnealingLR provides a smooth monotone LR decay from LR → eta_min, potentially allowing the model to find a flatter minimum and giving early stopping a cleaner signal.

> Everything else kept identical to C (LS=0.1, class_weights, Dropout=0.3).

In [ ]:
model_N     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_N = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_N = torch.optim.Adam(model_N.parameters(), lr=LR)
scheduler_N = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_N, T_max=EPOCHS, eta_min=1e-5)

model_N, history_N = run_experiment(model_N, "N_cosine_lr", criterion_N, optimizer_N, scheduler_N, loaders_base)

### O: C + WeightedRandomSampler + class weights (both imbalance corrections)
###### ──────────────────────────────────────────

> C corrects imbalance via loss weighting only.

> M corrects imbalance via sampler only (no loss weights — to avoid double-penalising minorities).

> Now test both together: sampler balances the batch distribution AND the loss keeps class weights to give minority classes extra gradient signal.

> Hypothesis: combining both corrections may help minority classes (Rock, Ground, Fighting) more than either alone, even at the risk of slight over-correction.

> Same architecture as C: MLP(512→256→128), Drop=0.3, LS=0.1.


In [ ]:
model_O     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_O = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)  # loss weights + sampler
optimizer_O = torch.optim.Adam(model_O.parameters(), lr=LR)
scheduler_O = torch.optim.lr_scheduler.StepLR(optimizer_O, step_size=5, gamma=0.5)

model_O, history_O = run_experiment(model_O, "O_c_sampler_cw", criterion_O, optimizer_O, scheduler_O, loaders_sampler)


### P: MLP with very light dropout (0.15) + winning C recipe
###### ──────────────────────────────────────────

> C wins with Drop=0.3. J tried Drop=0.2 (F1=0.2348, 2nd best LS model). B tried Drop=0.4 (0.2128).

> Pattern: lower dropout = better, within the LS models. Minimum tested so far is 0.2.

> Try Drop=0.15 — almost no dropout, but enough to prevent full memorisation of the 2880-sample dataset.

> Same winning recipe: MLP(512→256→128), LS=0.1, class_weights, StepLR.


In [ ]:
model_P     = MLP(img_size=IMG_SIZE, dropout=0.15).to(device)
criterion_P = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_P = torch.optim.Adam(model_P.parameters(), lr=LR)
scheduler_P = torch.optim.lr_scheduler.StepLR(optimizer_P, step_size=5, gamma=0.5)

print(f"P_drop015_ls params: {sum(p.numel() for p in model_P.parameters()):,}")
model_P, history_P = run_experiment(model_P, "P_drop015_ls", criterion_P, optimizer_P, scheduler_P, loaders_base)


### Q: WiderMLP + WeightedRandomSampler (no class weights in loss)
###### ──────────────────────────────────────────

> L used WiderMLP(1024) + class_weights in loss → similar result to C.

> M used MLP(512) + sampler (no loss weights) and was the best of L/M/N.

> This combines both: WiderMLP(1024) + sampler, no loss weights.

> Hypothesis: more capacity in the first layer + sampler-based balancing (without the double-correction risk of loss weights) may capture more colour combinations while staying balanced.

> Architecture: WiderMLP(12288→1024→256→128), Drop=0.3, LS=0.1, sampler.


In [ ]:
model_Q     = WiderMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_Q = nn.CrossEntropyLoss(label_smoothing=0.1)   # sampler handles imbalance
optimizer_Q = torch.optim.Adam(model_Q.parameters(), lr=LR)
scheduler_Q = torch.optim.lr_scheduler.StepLR(optimizer_Q, step_size=5, gamma=0.5)

print(f"Q_wider_sampler params: {sum(p.numel() for p in model_Q.parameters()):,}")
model_Q, history_Q = run_experiment(model_Q, "Q_wider_sampler", criterion_Q, optimizer_Q, scheduler_Q, loaders_sampler)


### R: C architecture + stronger label smoothing (LS=0.15)
###### ──────────────────────────────────────────

> C uses LS=0.1 and wins. Label smoothing is the single most impactful change across all experiments.

> LS=0.1 means: instead of target=[0,0,1,0,...], model sees [0.011,0.011,0.9,0.011,...] — 10% of confidence redistributed.

> LS=0.15 redistributes 15%: target becomes [0.017, ..., 0.85, ...].

> Hypothesis: harder classes (Rock=grey/brown, Fighting=humanoid) may benefit from more hedging. The model is less punished for assigning small probabilities to wrong classes, which could improve minority-class recall.

> Risk: too much smoothing may wash out the colour signal for easy classes (Fire=orange, Water=blue), hurting their F1.

> Same architecture: MLP(512→256→128), Drop=0.3, class_weights, StepLR.


In [ ]:
model_R     = MLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_R = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.15)
optimizer_R = torch.optim.Adam(model_R.parameters(), lr=LR)
scheduler_R = torch.optim.lr_scheduler.StepLR(optimizer_R, step_size=5, gamma=0.5)

model_R, history_R = run_experiment(model_R, "R_ls015_drop03", criterion_R, optimizer_R, scheduler_R, loaders_base)


### S: DeepMLP (4-layer funnel: 512→256→128→64) + winning C recipe
###### ──────────────────────────────────────────

> C uses MLP(3 hidden layers: 512→256→128). F used NarrowMLP (4 layers: 256→128→64→32) and failed (0.1617).

> F failed because its first layer was only 256-wide — too tight to learn from 12288 inputs.

> S addresses that: same 512-wide first layer as C (enough capacity for cross-pixel combos), then adds one extra compression step: 512→256→128→64.

> Hypothesis: one more narrow layer forces slightly more abstract representations before the final classifier.

> Risk: similar to F, extra depth may just slow learning without adding useful abstraction for flat pixel inputs.

> Architecture: DeepMLP(12288→512→256→128→64→9), Drop=0.3, LS=0.1, class_weights, StepLR.


In [ ]:
model_S     = DeepMLP(img_size=IMG_SIZE, dropout=0.3).to(device)
criterion_S = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)
optimizer_S = torch.optim.Adam(model_S.parameters(), lr=LR)
scheduler_S = torch.optim.lr_scheduler.StepLR(optimizer_S, step_size=5, gamma=0.5)

print(f"S_deep_ls params: {sum(p.numel() for p in model_S.parameters()):,}")
model_S, history_S = run_experiment(model_S, "S_deep_ls", criterion_S, optimizer_S, scheduler_S, loaders_base)


# Part 3 — Pre-Processing & Data Augmentation

> In this part we explore grayscale pre-processing and data augmentation. We run these **before** the final comparison (Part 4) so that all variants are included in the leaderboard and heatmap.


In [ ]:
# ── Find best solo RGB model so far ──────────────────────────────────────────
# Computed here (after Part 2) so Parts 3.1 and 3.2 can use the winning
# architecture for grayscale and augmentation experiments.
# Excludes ensembles (ENS_*), grayscale (*_gray, *_gray_eq), and already-augmented runs.
best_name = max(
    (k for k in results_tracker
     if not k.startswith("ENS_")
     and not k.endswith("_gray")
     and not k.endswith("_gray_eq")
     and not k.endswith("_augmented")),
    key=lambda k: results_tracker[k]["val_macro_f1"],
)
print(f"Best solo RGB model so far: {best_name}  "
      f"(val_macro_f1={results_tracker[best_name]['val_macro_f1']:.4f})")
print("Parts 3.1 and 3.2 will run this architecture with grayscale and augmentation.")


---
## 3.1 — Pre-Processing: Grayscale

**Scientific question:** Does colour information outweigh the curse-of-dimensionality penalty?

Pokémon type classification is strongly colour-dependent (Fire=orange, Water=blue, Poison=purple, Grass=green). Converting to grayscale eliminates the most discriminative signal — but it also cuts input dimensionality from **12 288 to 4 096** (3× smaller), which may help on our small dataset.

We expect colour to win — but we confirm it empirically.

**Two experiments on the current best solo RGB architecture:**
| ID | Name | Pre-processing | Notes |
|---|---|---|---|
| Gray | `{best_name}_gray` | plain grayscale | Same arch as best, `in_channels=1` |
| Gray+Eq | `{best_name}_gray_eq` | histogram equalise → grayscale | Stretches contrast before dropping colour |

**Grayscale pipeline:**
```
Raw PNG  →  [optional: PIL.ImageOps.equalize()]  →  Resize(64×64)  →  Grayscale(1)  →  ToTensor()  →  Normalize(mean=[0.5], std=[0.5])
```

| Variant | Output tensor shape | Flat input dim |
|---|---|---|
| Standard RGB | `(3, 64, 64)` | **12 288** |
| Plain grayscale | `(1, 64, 64)` | **4 096** |
| Gray + equalize | `(1, 64, 64)` | **4 096** |

Both experiments are registered in `model_registry` so they appear in the Part 4 heatmap and leaderboard automatically.


In [ ]:

# ── Gray: best model with plain grayscale input ───────────────────────────────
# Same architecture and hyperparams as the current best RGB model.
# Only change: in_channels=1 and grayscale loader. Input dim: 64*64*1 = 4096.
gray_name    = f"{best_name}_gray"
loaders_gray = build_loaders(grayscale=True, equalize=False)

model_gray     = model_registry[best_name]["model_gray"]().to(device)
criterion_gray = model_registry[best_name]["criterion"]()
optimizer_gray = torch.optim.Adam(model_gray.parameters(), lr=LR)
scheduler_gray = torch.optim.lr_scheduler.StepLR(optimizer_gray, step_size=5, gamma=0.5)

model_gray, history_gray = run_experiment(
    model_gray, gray_name, criterion_gray, optimizer_gray, scheduler_gray, loaders_gray,
)

# Register in model_registry so heatmap, leaderboard, and ensemble can use it
model_registry[gray_name] = {
    "model":      model_registry[best_name]["model_gray"],
    "criterion":  model_registry[best_name]["criterion"],
}
print(f"  Registered '{gray_name}' in model_registry.")

# ── Gray+Eq: same but with histogram equalisation before grayscale ────────────
gray_eq_name    = f"{best_name}_gray_eq"
loaders_gray_eq = build_loaders(grayscale=True, equalize=True)

model_gray_eq     = model_registry[best_name]["model_gray"]().to(device)
criterion_gray_eq = model_registry[best_name]["criterion"]()
optimizer_gray_eq = torch.optim.Adam(model_gray_eq.parameters(), lr=LR)
scheduler_gray_eq = torch.optim.lr_scheduler.StepLR(optimizer_gray_eq, step_size=5, gamma=0.5)

model_gray_eq, history_gray_eq = run_experiment(
    model_gray_eq, gray_eq_name, criterion_gray_eq, optimizer_gray_eq, scheduler_gray_eq, loaders_gray_eq,
)

model_registry[gray_eq_name] = {
    "model":     model_registry[best_name]["model_gray"],
    "criterion": model_registry[best_name]["criterion"],
}
print(f"  Registered '{gray_eq_name}' in model_registry.")

# ── Summary: RGB vs Grayscale ─────────────────────────────────────────────────
print(f"\n=== RGB vs Grayscale — {best_name} ===")
print(f"\n{'Variant':<30} {'val_F1':>8}  {'delta vs RGB':>12}")
print("-" * 55)
rgb_f1 = results_tracker[best_name]["val_macro_f1"]
print(f"{best_name:<30} {rgb_f1:>8.4f}  {'(baseline)':>12}")
for gn in [gray_name, gray_eq_name]:
    if gn not in results_tracker:
        continue
    gf1   = results_tracker[gn]["val_macro_f1"]
    delta = gf1 - rgb_f1
    print(f"{gn:<30} {gf1:>8.4f}  {delta:>+12.4f}")


---
## 3.2 — Data Augmentation Exploration

**Hypothesis:** Data augmentation (`RandomHorizontalFlip` + `ColorJitter`) should NOT meaningfully help an MLP — and we prove it empirically here.

**Why it doesn't help MLP (theoretical):**  
A `RandomHorizontalFlip` swaps pixel position (0, 0) with position (0, W-1). After flattening, the model sees a completely different 12 288-dimensional vector. The MLP has no concept of spatial layout, so a flipped image provides no useful training signal — it looks like a different unrelated example.  
`ColorJitter` adds slight colour variation, which could marginally help by forcing slightly less overfit to exact pixel values, but the effect is small.

**Why we do this anyway:**
1. Empirically confirms the theory — makes the Task 1 → Task 2 story clean: "augmentation doesn't help MLP; it will help CNN."
2. Validates that the augmentation pipeline (`get_augment_transforms`) works end-to-end.
3. Prepares the loader infrastructure for Task 2 and Task 3 where augmentation is critical.

We take the **current best solo RGB experiment** from Part 2 and retrain it with `augment=True`. Same model, same loss, same optimizer — only the training data loader changes.

#### Augmentation pipeline

```
Raw PNG  →  Resize(64, 64)  →  RandomHorizontalFlip(p=0.5)  →  ColorJitter(b=0.2, c=0.2, s=0.2)  →  RandomRotation(15°)  →  ToTensor()  →  Normalize
```
Applied to the **training loader only**; val/test always use the standard pipeline.  
Note: `RandomRotation` and `ColorJitter` are included here so the augmentation pipeline is complete for Task 2/3 CNN experiments — for MLP they are expected to have negligible or negative effect (confirmed below empirically).


In [ ]:
# ── Augmentation experiment — best model retrained with augment=True ─────────
# Only the training loader changes — model, criterion, optimizer are identical.
# Expected: val_macro_f1 roughly equal or slightly worse than without augmentation.

loaders_aug = build_loaders(augment=True, use_sampler=False)

# use model_registry — same factory as the current best, no if/elif needed
model_aug     = model_registry[best_name]["model"]().to(device)
criterion_aug = model_registry[best_name]["criterion"]()
optimizer_aug = torch.optim.Adam(model_aug.parameters(), lr=LR, weight_decay=1e-4)
scheduler_aug = torch.optim.lr_scheduler.StepLR(optimizer_aug, step_size=5, gamma=0.5)

aug_name = f"{best_name}_augmented"
model_aug, history_aug = run_experiment(
    model_aug, aug_name,
    criterion_aug, optimizer_aug, scheduler_aug, loaders_aug,
)

# Register augmented variant in model_registry so it appears in the heatmap,
# leaderboard plots, and any future ensemble comparisons — same factory as base model.
model_registry[aug_name] = {
    "model":     model_registry[best_name]["model"],
    "criterion": model_registry[best_name]["criterion"],
}
print(f"  Registered '{aug_name}' in model_registry.")

# compare augmented vs non-augmented side by side
f1_no_aug = results_tracker[best_name]["val_macro_f1"]
f1_aug    = results_tracker[aug_name]["val_macro_f1"]
delta     = f1_aug - f1_no_aug

print(f"\n=== Augmentation effect on best model ({best_name}) ===")
print(f"  Without augmentation : val_macro_f1 = {f1_no_aug:.4f}")
print(f"  With augmentation    : val_macro_f1 = {f1_aug:.4f}")
print(f"  Delta                : {delta:+.4f}  ({'improved' if delta > 0.005 else 'no meaningful gain' if abs(delta) <= 0.005 else 'slightly hurt'})")
print(f"\nConclusion: augmentation {'helps' if delta > 0.005 else 'does not meaningfully help'} MLP on this dataset.")
print("This is expected: MLP flattens the image, so a flipped image is an unrelated vector.")
print("Augmentation will be valuable for CNN (Task 2) where spatial structure is preserved.")


---
# Part 4 — Comparison, Final Model & Evaluation

Having all the experiments in the results tracker (including grayscale and augmented variants), let's analyse and compare all models.


## 4.1: Soft Ensemble Exploration
###### ──────────────────────────────────────────
> Lets start by cheking the f1 scores of each class of the best models we have to see if we can overlap and combine them.

> We can load some trained checkpoints, compute softmax probabilities for each, average them (soft vote), then pick the argmax.

> No retraining needed — this is pure inference-time combination.

> The idea is that maybe if our best model is bad in class A and B and another good model is good in those classes, so we can combine them


**Check Possible Good Ensemble Options**

> To make this analysis and selection of possible good ensembles check the plot bellow

In [ ]:
from src.evaluation.analysis import plot_per_class_f1_heatmap

# Build per-stem loader overrides dynamically:
# - stems ending in _gray_eq need build_loaders(grayscale=True, equalize=True)
# - stems ending in _gray (but not _gray_eq) need build_loaders(grayscale=True)
# - all other stems (RGB, augmented, extension) use the default build_loaders()
_loader_overrides = {}
for stem in model_registry:
    if stem.endswith("_gray_eq"):
        _loader_overrides[stem] = lambda: build_loaders(grayscale=True, equalize=True)
    elif stem.endswith("_gray"):
        _loader_overrides[stem] = lambda: build_loaders(grayscale=True)

fig = plot_per_class_f1_heatmap(
    checkpoint_dir     = TASK_OUT_DIR / "checkpoints",
    model_registry     = {k: v["model"] for k, v in model_registry.items()},
    loader_fn          = build_loaders,
    device             = device,
    out_path           = TASK_OUT_DIR / "plots" / "task1_per_class_f1_heatmap.png",
    highlight_best     = True,
    loader_fn_registry = _loader_overrides,
)
if fig:
    plt.show()
    plt.close(fig)


**Run Ensemble Combinations**

> Choose here the models you want to ensemble and check if we get better results with them

In [ ]:
from src.evaluation.ensemble import soft_ensemble, print_ensemble_report

# ── Configure the ensemble here — only thing you need to touch ────────────────
# names  : checkpoint stems (must exist in model_registry above)
# weights: None = uniform average; [0.6, 0.4] = weighted (auto-normalised)
ENS = {
    "names":   ["C_ls01_drop03", "E_sampler"],
    "weights": None,
}

# ── Run (automatic from here) ─────────────────────────────────────────────────
# build (model_instance, ckpt_path) pairs using model_registry — correct architecture guaranteed
ensemble_configs = [
    (model_registry[name]["model"](), TASK_OUT_DIR / "checkpoints" / f"{name}.pth")
    for name in ENS["names"]
]

missing = [str(p) for _, p in ensemble_configs if not p.exists()]
if missing:
    print(f"SKIP: missing checkpoints: {missing}")
else:
    ens_result = soft_ensemble(
        checkpoint_configs = ensemble_configs,
        val_loader         = build_loaders()[1],
        device             = device,
        weights            = ENS["weights"],
    )
    ens_label = "ENS_" + "_".join(ENS["names"])
    print_ensemble_report(ens_result, ensemble_label=ens_label)

    entry = {
        "val_macro_f1": ens_result["val_macro_f1"],
        "val_acc":      ens_result["val_acc"],
        "val_loss":     float("nan"),
        "total_epochs": 0,
        "train_time_s": 0.0,
        "history":      {},
        "child_models": ENS["names"],   # links back to individual experiments
    }
    results_tracker[ens_label] = entry
    _print_leaderboard(results_tracker)

    # persist immediately — same pattern as run_experiment
    save_experiment_result(ens_label, entry, RESULTS_PATH)

    # auto-update submission if this ensemble is the new overall best
    # compare against every other entry in results_tracker (solo + other ensembles)
    other_best = max(
        (v["val_macro_f1"] for k, v in results_tracker.items() if k != ens_label),
        default=-1,
    )
    if entry["val_macro_f1"] > other_best:
        print(f"\n  New overall best (ensemble) — regenerating submission CSV...")
        # ensemble members are always RGB — reuse global test_loader
        ens_test_configs = [
            (model_registry[n]["model"](), TASK_OUT_DIR / "checkpoints" / f"{n}.pth")
            for n in ENS["names"]
        ]
        ens_test_result = soft_ensemble(
            checkpoint_configs = ens_test_configs,
            val_loader         = test_loader,
            device             = device,
            weights            = ENS["weights"],
            inference_mode     = True,   # test loader returns uuid strings, not label tensors
        )
        generate_submission_from_preds(test_loader, ens_test_result["preds"], CLASSES, SUB_PATH)
        validate_submission(SUB_PATH)
        print(f"  Submission updated -> {SUB_PATH}")

    if SAVE_IN_EACH_RUN:
        save_to_drive(TASK_OUT_DIR, "task1", ROOT, IN_COLAB, DRIVE_OUTPUTS_DIR)


## 4.2: All Models Summary
###### ──────────────────────────────────────────

> See a **Leaderboard + bar chart** with all experiments sorted by `val_macro_f1`, so we can visualise and find gaps.


In [ ]:
from src.evaluation.analysis import plot_leaderboard

# printed leaderboard table (same as before)
print("=== All Experiments — Sorted by Val Macro-F1 ===\n")
_print_leaderboard(results_tracker)

# bar chart (leaderboard + training time) saved to outputs/plots/
fig = plot_leaderboard(
    results_tracker,
    out_path=TASK_OUT_DIR / "plots" / "task1_leaderboard.png",
)
plt.show()
plt.close(fig)

## 4.3: Best model deep dive
###### ──────────────────────────────────────────

> Reload the winning checkpoint, run the full classification report (per-class F1, precision, recall), and plot training curves + confusion matrix.


In [ ]:
from src.evaluation.analysis import print_classification_report
from src.evaluation.ensemble import soft_ensemble

# ── Helper: pick the right val/test loader for any model stem ─────────────────
def _get_loaders_for(stem: str):
    """Return (train_loader, val_loader) matching the input format of stem."""
    if stem.endswith("_gray_eq"):
        return build_loaders(grayscale=True, equalize=True)
    elif stem.endswith("_gray"):
        return build_loaders(grayscale=True)
    else:
        return build_loaders()

def _get_test_loader_for(stem: str):
    """Return a test DataLoader with the correct transform for stem."""
    from src.datasets.dataset import PokemonDataset, get_base_transforms, get_gray_transforms
    if stem.endswith("_gray_eq"):
        t = get_gray_transforms(IMG_SIZE, equalize=True)
    elif stem.endswith("_gray"):
        t = get_gray_transforms(IMG_SIZE, equalize=False)
    else:
        t = get_base_transforms(IMG_SIZE)
    ds = PokemonDataset(TEST_DIR, t, csv_path=None)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

# Overall best (could be an ensemble) — used for final test predictions + submission
overall_best_name = max(results_tracker, key=lambda k: results_tracker[k]["val_macro_f1"])
print(f"Overall best    : {overall_best_name}  (val_macro_f1={results_tracker[overall_best_name]['val_macro_f1']:.4f})")

# Best SOLO MODEL — for the history plot, confusion matrix, classification report.
# Excludes only ensembles (ENS_*); Gray, augmented, and extension models are all valid winners.
best_name = max(
    (k for k in results_tracker if not k.startswith("ENS_")),
    key=lambda k: results_tracker[k]["val_macro_f1"],
)
print(f"Best solo model : {best_name}  (val_macro_f1={results_tracker[best_name]['val_macro_f1']:.4f})")

# ── Deep dive on best solo model ──────────────────────────────────────────────
best_model     = model_registry[best_name]["model"]().to(device)
best_criterion = model_registry[best_name]["criterion"]()
best_model.load_state_dict(
    torch.load(TASK_OUT_DIR / "checkpoints" / f"{best_name}.pth", map_location=device, weights_only=True)
)

# Use the loader that matches the best model's input format (RGB, gray, or gray+eq)
val_loader_final = _get_loaders_for(best_name)[1]

all_labels_list, all_preds = print_classification_report(best_model, val_loader_final, device, CLASSES)
val_metrics_final = evaluate(best_model, val_loader_final, best_criterion, device)
print(f"\nval_loss={val_metrics_final['loss']:.4f}  val_acc={val_metrics_final['acc']:.4f}  macro_f1={val_metrics_final['macro_f1']:.4f}")

# ── Test predictions from overall best ───────────────────────────────────────
# Two cases: solo model (any type — RGB/gray/augmented/extension) or ensemble.
if not overall_best_name.startswith("ENS_"):
    # solo model — use the correct test loader for its input format
    ov_model = model_registry[overall_best_name]["model"]().to(device)
    ov_model.load_state_dict(
        torch.load(TASK_OUT_DIR / "checkpoints" / f"{overall_best_name}.pth", map_location=device, weights_only=True)
    )
    ov_model.eval()
    ov_test_loader = _get_test_loader_for(overall_best_name)
    all_test_preds = []
    with torch.no_grad():
        for imgs, _ in ov_test_loader:
            imgs = imgs.to(device)
            all_test_preds.extend(ov_model(imgs).argmax(dim=1).cpu().tolist())

else:
    # overall best is an ensemble — re-run soft_ensemble on test_loader
    # inference_mode=True: test set has no integer labels (returns uuid strings)
    member_names = results_tracker[overall_best_name].get("child_models", [])
    if not member_names:
        member_names = overall_best_name[len("ENS_"):].split("_")
    print(f"\nEnsemble members: {member_names}")
    ens_test_configs = [
        (model_registry[n]["model"](), TASK_OUT_DIR / "checkpoints" / f"{n}.pth")
        for n in member_names
    ]
    ens_test_result = soft_ensemble(
        checkpoint_configs = ens_test_configs,
        val_loader         = test_loader,
        device             = device,
        weights            = None,
        inference_mode     = True,   # test loader returns uuid strings, not label tensors
    )
    all_test_preds = ens_test_result["preds"]
    ov_test_loader = test_loader   # ensemble members are always RGB
    print(f"Ensemble test predictions collected ({len(all_test_preds)} images).")

# ── Auto-save full results snapshot + regenerate submission ───────────────────
save_all_results(
    results_tracker = results_tracker,
    best_name       = best_name,
    val_metrics     = val_metrics_final,
    all_labels      = all_labels_list,
    all_preds       = all_preds,
    config          = {
        "FAST_RUN": FAST_RUN, "EPOCHS": EPOCHS, "LR": LR,
        "BATCH_SIZE": BATCH_SIZE, "PATIENCE": PATIENCE, "IMG_SIZE": IMG_SIZE,
    },
    results_path    = RESULTS_PATH,
)

# Submission: always re-run here so the final CSV matches overall_best_name.
generate_submission_from_preds(ov_test_loader, all_test_preds, CLASSES, SUB_PATH)
validate_submission(SUB_PATH, expected_rows=900)
print(f"Overall best used : {overall_best_name}  (val_macro_f1={results_tracker[overall_best_name]['val_macro_f1']:.4f})")


In [ ]:
# ── Training curves + confusion matrix for best experiment ────────────────────
best_history = results_tracker[best_name]["history"]

fig = plot_history(best_history, TASK_OUT_DIR / "plots" / "task1_history.png", title=best_name)
plt.show(); plt.close(fig)

fig = plot_confusion_matrix(all_labels_list, all_preds, CLASSES, TASK_OUT_DIR / "plots" / "task1_confusion.png")
plt.show(); plt.close(fig)


---
# Final Summary & Submission

### Run results

| Metric | Value |
|---|---|
| Best experiment (overall) | ENS_C_ls01_drop03_E_sampler |
| Best val macro-F1 (overall) | **0.2428** (ensemble: C + E) |
| Best val accuracy (overall) | 0.2667 |
| Best experiment (solo) | R_ls015_drop03 |
| Best val macro-F1 (solo) | **0.2396** (MLP, LS=0.15, Drop=0.3, class_weights) |
| Kaggle public score | **0.2288** (submitted ENS_C_ls01_drop03_E_sampler) |
| Epochs run (best solo) | 32 (early stopping, patience=7) |
| Total experiment time | ~2 658 s (~44 min) across 19 solo experiments |

### Experiments run

| ID | Name | Architecture | Part | Monitor | val_F1 | Notes |
|---|---|---|---|---|---|---|
| A | vanilla | VanillaMLP (128→64), no BN/Drop | 2 | val_macro_f1 | 0.2203 | Bare baseline |
| B | mlp_base | MLP (512→256→128), Drop=0.4 | 2 | val_macro_f1 | 0.2222 | First Colab run equivalent |
| C | ls01_drop03 | MLP, Drop=0.3, LS=0.1, CW | 2 | val_macro_f1 | 0.2395 | Best regularised baseline |
| D | wd1e4 | MLP, Drop=0.3, WD=1e-4 | 2 | val_macro_f1 | 0.1860 | WD hurts with imbalanced data |
| E | sampler | MLP, Drop=0.3, WeightedSampler | 2 | val_macro_f1 | 0.2357 | Sampler improves minority classes |
| F | narrow | NarrowMLP (256→128→64→32) | 2 | val_macro_f1 | 0.1739 | Too narrow — underfits |
| G | bottleneck | BottleneckMLP (512→1024→256→128) | 2 | val_macro_f1 | 0.2285 | Wide middle helps slightly |
| H | vanilla_v2 | VanillaMLP_v2 (256→128), no reg | 2 | val_macro_f1 | 0.2072 | More capacity, still weaker |
| I | v2_rock_weights | VanillaMLP_v2, CE(Rock×3, Ground×2) | 2 | val_macro_f1 | 0.2116 | Targeted upweighting marginal |
| J | mlp_drop02 | MLP, Drop=0.2 | 2 | val_macro_f1 | 0.2360 | Lighter dropout competitive |
| K | v2_wd1e5 | VanillaMLP_v2, WD=1e-5 | 2 | val_macro_f1 | 0.2124 | Minimal WD, no gain |
| L | wider_ls | WiderMLP (1024→512→256), LS=0.1 | 2 | val_macro_f1 | 0.2196 | Wider hurts — more overfitting |
| M | c_sampler | MLP (C arch) + WeightedSampler | 2 | val_macro_f1 | 0.2361 | Sampler on top of C — ~same as E |
| N | cosine_lr | MLP (C arch) + CosineAnnealing | 2 | val_macro_f1 | 0.2251 | Cosine schedule, no improvement |
| O | c_sampler_cw | MLP (C+Sampler+CW together) | 2 | val_macro_f1 | 0.1920 | CW + sampler double-compensate, hurts |
| P | drop015_ls | MLP, Drop=0.15, LS=0.1 | 2 | val_macro_f1 | 0.2171 | Less dropout, marginal |
| Q | wider_sampler | WiderMLP + WeightedSampler | 2 | val_macro_f1 | 0.2374 | Wider + sampler helps capacity |
| **R** | **ls015_drop03** | **MLP, Drop=0.3, LS=0.15, CW** | **2** | val_macro_f1 | **0.2396** | **Best solo — LS=0.15 > LS=0.10** |
| S | deep_ls | DeepMLP (4-layer funnel), LS=0.1 | 2 | val_macro_f1 | 0.1905 | Deeper hurts with small dataset |
| Gray_A | vanilla (gray) | VanillaMLP(in_channels=1) | 3.1 | val_macro_f1 | 0.1362 | Grayscale kills color signal |
| Gray_B | eq_mlp (gray+eq) | MLP(drop=0.3, in_channels=1) | 3.1 | val_macro_f1 | 0.1531 | Equalisation helps slightly |
| R_aug | ls015_drop03_augmented | Best arch + augment=True | 3.2 | val_macro_f1 | 0.1927 | Augmentation hurts MLP (no spatial bias) |
| **ENS_C_E** | **ENS_C_ls01_drop03_E_sampler** | **Soft avg C + E** | **4.1** | — | **0.2428** | **Best overall — submitted to Kaggle** |
| ENS_R_E_P | ENS_R_ls015_drop03_E_sampler_P | Soft avg R + E + P | 4.1 | — | 0.2427 | Close second (3-model) |
| ENS_C_E_P | ENS_C_ls01_drop03_E_sampler_P | Soft avg C + E + P | 4.1 | — | 0.2405 | Third best |

### Confirmed findings

1. **Regularisation effect:** C (LS=0.1, Drop=0.3) and R (LS=0.15, Drop=0.3) dominate the leaderboard. Heavier WD (D) hurts badly. Lighter dropout (J) is competitive but not better. The sweet spot for this tiny dataset is mild label smoothing + dropout, no weight decay.
2. **Augmentation effect:** `RandomHorizontalFlip` **hurts** MLP (R_augmented: 0.1927 vs R: 0.2396). Confirmed: flipping produces a completely different 12 288-vector — MLP has no spatial invariance to benefit from it.
3. **Grayscale effect:** Grayscale experiments (Gray_A: 0.1362, Gray_B: 0.1531) score far below RGB counterparts. Colour is the primary discriminative signal (Fire=orange/red, Water=blue). Histogram equalisation recovers some signal on dark sprites.
4. **Hard classes:** Rock and Ground have the lowest per-class F1. Their grey/brown colour profiles overlap with each other and with Fighting/Normal. The confusion matrix confirms Rock ↔ Ground is the dominant error.
5. **MLP ceiling:** Val macro-F1 plateaued at ~0.24 regardless of architecture/regularisation choices. The fundamental bottleneck is lack of spatial structure — CNN (Task 2) will break through this by preserving pixel neighbourhood relationships.
6. **Ensemble gain:** The best 2-model ensemble (C + E) improves over best solo by +0.0032 F1 (0.2428 vs 0.2396). C and E are complementary: C uses class weights, E uses WeightedRandomSampler — different strategies to handle class imbalance. Adding a third model (R or P) did not further improve.

### Infrastructure built (`src/`)

- `src/models/mlp.py` — 7 MLP classes: MLP, VanillaMLP, VanillaMLP_v2, NarrowMLP, WiderMLP, BottleneckMLP, DeepMLP — all with `in_channels` param (RGB=3 or grayscale=1)
- `src/datasets/dataset.py` — RGB, augmented, grayscale, and gray+equalize transform pipelines; `grayscale`/`equalize` params in loaders
- `src/training/early_stopping.py` — metric-agnostic, monitors by minimisation (`stopper(-val_f1, model)`)
- `src/evaluation/metrics.py`, `plots.py`, `submission.py` — classification report, training curves, confusion matrix, submission CSV
- `src/evaluation/ensemble.py` — soft ensemble with `inference_mode` for test-set inference (test labels are uuid strings, not ints)
- `src/evaluation/persistence.py` — `save_to_drive` / `restore_from_drive` for automatic Drive sync on Colab
- Tests: `models_test.py` (14), `dataset_test.py` (13), `train_test.py` (3) — all pass locally


In [ ]:
# ── Save outputs zip to Drive (Colab only) ───────────────────────────────────
# Zips task1/outputs/ and copies it to Google Drive.
# On local runs just prints the output path — safe to call anywhere.
save_to_drive(TASK_OUT_DIR, "task1", ROOT, IN_COLAB, DRIVE_OUTPUTS_DIR)
